In [ ]:
# import tensorflow as tf

# # Check if TensorFlow can access a GPU
# if tf.config.list_physical_devices('GPU'):
#     print("GPU is available.")
# else:
#     print("GPU is not available.")


selecting png and converting them to npy

In [ ]:
# import os
# import numpy as np
# from PIL import Image

# def save_pngs_to_npy(input_folder):
#     image_arrays = [
#         np.array(Image.open(os.path.join(input_folder, f)).convert('RGB'))
#         for f in os.listdir(input_folder) if f.lower().endswith(".png")
#     ]

#     if not image_arrays:
#         raise ValueError("No PNG files found in the specified folder.")

#     np.save(os.path.join(input_folder, "11000_noise_rotation_aug.npy"), np.stack(image_arrays, axis=0))

# # Use your specified path
# input_folder = "/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/11000_noise_rotation_aug"

# save_pngs_to_npy(input_folder)


In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import tqdm
import tensorflow as tf
from tensorflow.keras.models import Model

from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, Cropping2D,UpSampling2D
from tensorflow.keras.layers import Conv2DTranspose, ZeroPadding2D

2026-03-02 11:51:59.308867: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-02 11:51:59.546270: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 11:52:00.328890: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
noisy_images1 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/input/00000.npy")
noisy_images2 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/input/05000_wrinkles.npy")
noisy_images3 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/input/12000_noise_rotation_aug.npy")
noisy_images4 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/input/16000_images.npy")
noisy_images5 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/11000_noise_rotation_aug.npy")

In [3]:
ground_truth_images1 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/GT/00000.npy")
ground_truth_images2 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/GT/05000.npy")
ground_truth_images3 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/GT/12000_50DPI.npy")
ground_truth_images4 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/GT/16000.npy")
ground_truth_images5 = np.load("/home/user/Desktop/Anindita_digitization/noise and rotation/New folder/11000_50DPI.npy")

In [4]:
noisy_images1.shape,noisy_images2.shape,noisy_images3.shape,noisy_images4.shape,noisy_images5.shape

((987, 425, 550, 3),
 (999, 425, 550, 3),
 (1000, 425, 550, 3),
 (1000, 425, 550, 3),
 (995, 425, 550, 3))

In [5]:
ground_truth_images1.shape,ground_truth_images2.shape,ground_truth_images3.shape,ground_truth_images4.shape,ground_truth_images5.shape

((987, 425, 550, 3),
 (999, 425, 550, 3),
 (1000, 425, 550, 3),
 (1000, 425, 550, 3),
 (995, 425, 550, 3))

processing the images

In [6]:
def preprocess_images(images):
    processed_images = []
    for image in images:
        hsv_image = cv2.cvtColor(image, cv2.COLOR_RGB2HSV)
        value_channel = hsv_image[:, :, 2]
        processed_images.append(value_channel)
    return np.array(processed_images)


processed_ip_images1 = preprocess_images(noisy_images1)
processed_ip_images2 = preprocess_images(noisy_images2)
processed_ip_images3 = preprocess_images(noisy_images3)
processed_ip_images4 = preprocess_images(noisy_images4)
processed_ip_images5 = preprocess_images(noisy_images5)

processed_gt_images1 = preprocess_images(ground_truth_images1)
processed_gt_images2 = preprocess_images(ground_truth_images2)
processed_gt_images3 = preprocess_images(ground_truth_images3)
processed_gt_images4 = preprocess_images(ground_truth_images4)
processed_gt_images5 = preprocess_images(ground_truth_images5)


In [7]:
processed_ip_images1.shape,processed_ip_images2.shape,processed_ip_images3.shape,processed_ip_images4.shape,processed_ip_images5.shape

((987, 425, 550),
 (999, 425, 550),
 (1000, 425, 550),
 (1000, 425, 550),
 (995, 425, 550))

In [8]:
processed_gt_images1.shape,processed_gt_images2.shape,processed_gt_images3.shape,processed_gt_images4.shape,processed_gt_images5.shape

((987, 425, 550),
 (999, 425, 550),
 (1000, 425, 550),
 (1000, 425, 550),
 (995, 425, 550))

In [9]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate, ZeroPadding2D
from tensorflow.keras.models import Model
import tensorflow as tf
from tensorflow.keras.layers import Conv2D, MaxPooling2D, BatchNormalization, Flatten, Dense, Dropout
import math

class SpatialTransformer(tf.keras.layers.Layer):
    def __init__(self, name="spatial_transformer", sigma=1.0, kernel_size=3):
        """
        Spatial Transformer Network (STN) with Gaussian kernel-based interpolation.
        Args:
            sigma: Standard deviation of Gaussian kernel.
            kernel_size: Radius of Gaussian window (half-width). Common: 2 or 3.
        """
        super(SpatialTransformer, self).__init__(name=name)
        self.sigma = sigma
        self.kernel_size = kernel_size

        # --- Coarse localization network ---
        self.localization_net = tf.keras.Sequential([
            Conv2D(16, kernel_size=7, activation='relu', padding='same'),
            BatchNormalization(),
            Conv2D(32, kernel_size=5, activation='relu', padding='same'),
            BatchNormalization(),
            MaxPooling2D(pool_size=(2, 2), padding='same'),
            Conv2D(64, kernel_size=3, activation='relu', padding='same'),
            BatchNormalization(),
            MaxPooling2D(pool_size=(2, 2), padding='same'),
            Flatten(),
            Dense(100, activation='relu'),
            Dropout(0.3),
            Dense(6, activation=None,
                  kernel_initializer='zeros',
                  bias_initializer=tf.constant_initializer([1, 0, 0, 0, 1, 0]))
        ])

        # --- Fine localization network ---
        self.localization_net_fine = tf.keras.Sequential([
            Conv2D(8, kernel_size=5, activation='relu', padding='same'),
            BatchNormalization(),
            MaxPooling2D(pool_size=(2, 2), padding='same'),
            Flatten(),
            Dense(50, activation='relu'),
            Dense(6, activation=None,
                  kernel_initializer='zeros',
                  bias_initializer=tf.constant_initializer([1, 0, 0, 0, 1, 0]))
        ])

        # Learnable scalar for adaptive Gaussian weighting
        self.kernel_weights = tf.Variable(
            initial_value=tf.ones((1, 1, 1, 1), dtype=tf.float32),
            trainable=True,
            name='learnable_kernel_weight'
        )

    def call(self, x):
        """Forward pass with coarse and fine Gaussian warping."""
        # --- Coarse transformation ---
        affine_params = self.localization_net(x)
        coarse_affine = tf.reshape(affine_params, (-1, 2, 3))
        grid_coarse = self._affine_grid_generator(coarse_affine, tf.shape(x)[1], tf.shape(x)[2])
        x_warped_coarse = self._gaussian_sampler(x, grid_coarse)

        # --- Fine transformation ---
        affine_params_fine = self.localization_net_fine(x_warped_coarse)
        fine_affine = tf.reshape(affine_params_fine, (-1, 2, 3))
        grid_fine = self._affine_grid_generator(fine_affine, tf.shape(x)[1], tf.shape(x)[2])
        x_warped_fine = self._gaussian_sampler(x_warped_coarse, grid_fine)

        return x_warped_fine

    # -----------------------------------------------------
    #  Affine transformation grid generator
    # -----------------------------------------------------
    def _affine_grid_generator(self, theta, height, width):
        num_batch = tf.shape(theta)[0]
        x = tf.linspace(-1.0, 1.0, width)
        y = tf.linspace(-1.0, 1.0, height)
        x_t, y_t = tf.meshgrid(x, y)
        ones = tf.ones_like(x_t)
        sampling_grid = tf.stack([x_t, y_t, ones], axis=0)
        sampling_grid = tf.expand_dims(sampling_grid, axis=0)
        sampling_grid = tf.tile(sampling_grid, [num_batch, 1, 1, 1])
        sampling_grid = tf.reshape(sampling_grid, [num_batch, 3, -1])
        grid = tf.matmul(theta, sampling_grid)
        grid = tf.reshape(grid, [num_batch, 2, height, width])
        return tf.transpose(grid, [0, 2, 3, 1])  # [B,H,W,2]

    # -----------------------------------------------------
    #  Gaussian Kernel Sampler (differentiable)
    # -----------------------------------------------------
    def _gaussian_sampler(self, img, grid):
        """
        Differentiable Gaussian interpolation sampler.
        img: [B,H,W,C]
        grid: normalized coordinates [-1,1], shape [B,H,W,2]
        """
        k = self.kernel_size
        sigma = self.sigma

        B, H, W, C = tf.unstack(tf.shape(img))
        x = (grid[..., 0] + 1.0) * 0.5 * tf.cast(W - 1, tf.float32)
        y = (grid[..., 1] + 1.0) * 0.5 * tf.cast(H - 1, tf.float32)

        x_floor = tf.floor(x)
        y_floor = tf.floor(y)

        # Offsets in neighborhood
        offsets = tf.range(-k, k + 1, dtype=tf.float32)
        x_offsets = tf.reshape(offsets, [1, 1, 1, -1, 1])  # [1,1,1,2k+1,1]
        y_offsets = tf.reshape(offsets, [1, 1, 1, 1, -1])  # [1,1,1,1,2k+1]

        x_neigh = x_floor[..., None, None] + x_offsets
        y_neigh = y_floor[..., None, None] + y_offsets

        x_neigh = tf.clip_by_value(x_neigh, 0.0, tf.cast(W - 1, tf.float32))
        y_neigh = tf.clip_by_value(y_neigh, 0.0, tf.cast(H - 1, tf.float32))

        x_int = tf.cast(x_neigh, tf.int32)
        y_int = tf.cast(y_neigh, tf.int32)

        # Batch index broadcasting
        batch_idx = tf.reshape(tf.range(B), [B, 1, 1, 1, 1])
        batch_idx = tf.tile(batch_idx, [1, H, W, 2 * k + 1, 2 * k + 1])

        y_int = tf.broadcast_to(y_int, [B, H, W, 2 * k + 1, 2 * k + 1])
        x_int = tf.broadcast_to(x_int, [B, H, W, 2 * k + 1, 2 * k + 1])

        indices = tf.stack([batch_idx, y_int, x_int], axis=-1)
        pixels = tf.gather_nd(img, indices)

        # Gaussian weights
        dx = x[..., None, None] - x_neigh
        dy = y[..., None, None] - y_neigh
        weights = tf.exp(-0.5 * (dx**2 + dy**2) / (sigma**2))
        weights_sum = tf.reduce_sum(weights, axis=[-2, -1], keepdims=True)
        weights /= (weights_sum + 1e-12)

        # Weighted sum
        weighted = pixels * tf.expand_dims(weights, axis=-1)
        out = tf.reduce_sum(weighted, axis=[-2, -3])

        # Apply learnable scale
        out *= self.kernel_weights
        return out



from tensorflow.keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, ZeroPadding2D, Input, concatenate, BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras import regularizers

def unet_with_stn(input_shape=(425, 550, 1), dropout_rate=0.2):
    inputs = Input(input_shape)

    # Encoder
    c1 = Conv2D(16, (3, 3), activation='relu', padding='same')(inputs)
    c1 = BatchNormalization()(c1)
    c1 = Dropout(dropout_rate)(c1)
    c1 = Conv2D(16, (3, 3), activation='relu', padding='same')(c1)
    c1 = BatchNormalization()(c1)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = Conv2D(32, (3, 3), activation='relu', padding='same')(p1)
    c2 = BatchNormalization()(c2)
    c2 = Dropout(dropout_rate)(c2)
    c2 = Conv2D(32, (3, 3), activation='relu', padding='same')(c2)
    c2 = BatchNormalization()(c2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = Conv2D(64, (3, 3), activation='relu', padding='same')(p2)
    c3 = BatchNormalization()(c3)
    c3 = Dropout(dropout_rate)(c3)
    c3 = Conv2D(64, (3, 3), activation='relu', padding='same')(c3)
    c3 = BatchNormalization()(c3)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = Conv2D(128, (3, 3), activation='relu', padding='same')(p3)
    c4 = BatchNormalization()(c4)
    c4 = Dropout(dropout_rate)(c4)
    c4 = Conv2D(128, (3, 3), activation='relu', padding='same')(c4)
    c4 = BatchNormalization()(c4)
    p4 = MaxPooling2D((2, 2))(c4)

    # Bottleneck with STN
    c5 = Conv2D(256, (3, 3), activation='relu', padding='same')(p4)
    c5 = BatchNormalization()(c5)
    c5 = Dropout(dropout_rate)(c5)
    c5 = Conv2D(256, (3, 3), activation='relu', padding='same')(c5)
    c5 = BatchNormalization()(c5)

    # Spatial Transformer
    x = SpatialTransformer(sigma=1.0, kernel_size=3)(c5)

    # Decoder
    u6 = Conv2DTranspose(128, (2, 2), strides=(2, 2), padding='same')(x)
    u6 = ZeroPadding2D(((0, max(0, c4.shape[1] - u6.shape[1])),
                        (0, max(0, c4.shape[2] - u6.shape[2]))))(u6)
    u6 = concatenate([u6, c4])

    c6 = Conv2D(128, (3, 3), activation='relu', padding='same')(u6)
    c6 = BatchNormalization()(c6)
    c6 = Dropout(dropout_rate)(c6)
    c6 = Conv2D(128, (3, 3), activation='relu', padding='same')(c6)
    c6 = BatchNormalization()(c6)

    u7 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c6)
    u7 = ZeroPadding2D(((0, max(0, c3.shape[1] - u7.shape[1])),
                        (0, max(0, c3.shape[2] - u7.shape[2]))))(u7)
    u7 = concatenate([u7, c3])

    c7 = Conv2D(64, (3, 3), activation='relu', padding='same')(u7)
    c7 = BatchNormalization()(c7)
    c7 = Dropout(dropout_rate)(c7)
    c7 = Conv2D(64, (3, 3), activation='relu', padding='same')(c7)
    c7 = BatchNormalization()(c7)

    u8 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c7)
    u8 = ZeroPadding2D(((0, max(0, c2.shape[1] - u8.shape[1])),
                        (0, max(0, c2.shape[2] - u8.shape[2]))))(u8)
    u8 = concatenate([u8, c2])

    c8 = Conv2D(32, (3, 3), activation='relu', padding='same')(u8)
    c8 = BatchNormalization()(c8)
    c8 = Dropout(dropout_rate)(c8)
    c8 = Conv2D(32, (3, 3), activation='relu', padding='same')(c8)
    c8 = BatchNormalization()(c8)

    u9 = Conv2DTranspose(16, (2, 2), strides=(2, 2), padding='same')(c8)
    u9 = ZeroPadding2D(((0, max(0, c1.shape[1] - u9.shape[1])),
                        (0, max(0, c1.shape[2] - u9.shape[2]))))(u9)
    u9 = concatenate([u9, c1])

    c9 = Conv2D(16, (3, 3), activation='relu', padding='same')(u9)
    c9 = BatchNormalization()(c9)
    c9 = Dropout(dropout_rate)(c9)
    c9 = Conv2D(16, (3, 3), activation='relu', padding='same')(c9)
    c9 = BatchNormalization()(c9)

    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c9)

    model = Model(inputs=inputs, outputs=outputs)
    return model


input_shape = (425, 550, 1)  # Match STN input shape

model = unet_with_stn()
model.summary()


model1 = unet_with_stn()
model2 = unet_with_stn()
model3 = unet_with_stn()
model4 = unet_with_stn()

2026-03-02 11:52:08.677482: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1639] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 14400 MB memory:  -> device: 0, name: NVIDIA RTX A4000, pci bus id: 0000:19:00.0, compute capability: 8.6


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 425, 550, 1)]        0         []                            
                                                                                                  
 conv2d (Conv2D)             (None, 425, 550, 16)         160       ['input_1[0][0]']             
                                                                                                  
 batch_normalization (Batch  (None, 425, 550, 16)         64        ['conv2d[0][0]']              
 Normalization)                                                                                   
                                                                                                  
 dropout (Dropout)           (None, 425, 550, 16)         0         ['batch_normalization[0][0

In [10]:
from sklearn.model_selection import train_test_split
train_noisy1, val_noisy1, train_clean1, val_clean1 = train_test_split(processed_ip_images1,processed_gt_images1, test_size=0.1, random_state=42)

In [11]:
from sklearn.model_selection import train_test_split
train_noisy2, val_noisy2, train_clean2, val_clean2 = train_test_split(processed_ip_images2,processed_gt_images2, test_size=0.1, random_state=42)

In [12]:
from sklearn.model_selection import train_test_split
train_noisy3, val_noisy3, train_clean3, val_clean3 = train_test_split(processed_ip_images3,processed_gt_images3, test_size=0.1, random_state=42)

In [13]:
from sklearn.model_selection import train_test_split
train_noisy4, val_noisy4, train_clean4, val_clean4 = train_test_split(processed_ip_images4,processed_gt_images4, test_size=0.1, random_state=42)

In [14]:
from sklearn.model_selection import train_test_split
train_noisy5, val_noisy5, train_clean5, val_clean5 = train_test_split(processed_ip_images5,processed_gt_images5, test_size=0.1, random_state=42)

In [15]:
def process_set(train_noisy, train_clean, val_noisy, val_clean):
    # Normalize
    train_noisy = train_noisy / 255.0
    train_clean = train_clean / 255.0
    val_noisy = val_noisy / 255.0
    val_clean = val_clean / 255.0

    # Add channel dimension
    train_noisy = train_noisy[..., tf.newaxis]
    train_clean = train_clean[..., tf.newaxis]
    val_noisy = val_noisy[..., tf.newaxis]
    val_clean = val_clean[..., tf.newaxis]

    return train_noisy, train_clean, val_noisy, val_clean


train_noisy1, train_clean1, val_noisy1, val_clean1 = process_set(
    train_noisy1, train_clean1, val_noisy1, val_clean1
)


train_noisy2, train_clean2, val_noisy2, val_clean2 = process_set(
    train_noisy2, train_clean2, val_noisy2, val_clean2
)


train_noisy3, train_clean3, val_noisy3, val_clean3 = process_set(
    train_noisy3, train_clean3, val_noisy3, val_clean3
)


train_noisy4, train_clean4, val_noisy4, val_clean4 = process_set(
    train_noisy4, train_clean4, val_noisy4, val_clean4
)


train_noisy5, train_clean5, val_noisy5, val_clean5 = process_set(
    train_noisy5, train_clean5, val_noisy5, val_clean5
)


In [16]:
# train_noisy.shape, val_noisy.shape, train_clean.shape, val_clean.shape

In [17]:
import tensorflow as tf
from tensorflow.keras import backend as K



# --- Dice Loss ---
def dice_loss(y_true, y_pred):
    smooth = 1.0
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    dice = (2. * intersection + smooth) / (K.sum(y_true_f) + K.sum(y_pred_f) + smooth)
    return 1 - dice


# --- Focal Loss ---
def focal_loss(y_true, y_pred, gamma=2.0, alpha=0.25):
    """
    Focal loss for binary segmentation.
    gamma: focusing parameter (higher = more focus on hard examples)
    alpha: class balance factor
    """
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(y_pred, K.epsilon(), 1.0 - K.epsilon())

    cross_entropy = - (y_true * tf.math.log(y_pred) + (1 - y_true) * tf.math.log(1 - y_pred))
    weight = alpha * y_true * tf.pow(1 - y_pred, gamma) + (1 - alpha) * (1 - y_true) * tf.pow(y_pred, gamma)
    loss = weight * cross_entropy
    return tf.reduce_mean(loss)


# --- MS-SSIM Loss ---
def ms_ssim_loss(y_true, y_pred):
    # tf.image.ssim_multiscale returns values in [0,1], higher is better
    return 1 - tf.reduce_mean(tf.image.ssim_multiscale(y_true, y_pred, max_val=1.0))


# --- Combined Loss (Your Formula with Focal Instead of Tversky) ---
def custom_loss(y_true, y_pred):
    ms_ssim = 1 - ms_ssim_loss(y_true, y_pred)  # actual MS-SSIM value
    dice = 1 - dice_loss(y_true, y_pred)        # actual Dice value
    focal = 1 - focal_loss(y_true, y_pred)      # actual Focal similarity (conceptually)

    # L = 0.4(1−MS-SSIM) + 0.35(1−Dice) + 0.25(1−Focal)
    return (
        0.5 * (1 - ms_ssim) +
        0.25 * (1 - dice) +
        0.25 * (1 - focal)
    )


# --- SSIM Metric for Evaluation ---
def ssim_metric(y_true, y_pred):
    return tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))

def dice_coefficient(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)



In [22]:

# model1.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss=custom_loss,
#     metrics=[ssim_metric, dice_coefficient]
# )


# # Train the model
# history = model1.fit(
#     train_noisy1, train_clean1,
#     validation_data=(val_noisy1, val_clean1),
#     batch_size=8,
#     epochs=100,
#     verbose=1
# )



Epoch 1/100


2026-03-02 11:53:10.323526: E tensorflow/core/grappler/optimizers/meta_optimizer.cc:954] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inmodel_1/dropout_10/dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
2026-03-02 11:53:11.667756: I tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:432] Loaded cuDNN version 8600
2026-03-02 11:53:12.249820: I tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:606] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-03-02 11:53:13.570878: I tensorflow/compiler/xla/service/service.cc:168] XLA service 0x7f5c35392f20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-03-02 11:53:13.570906: I tensorflow/compiler/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA RTX A4000, Compute Capability 8.6
2026-03-02 11:53:13.582404: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:25

111/111 [==============================] - 55s 270ms/step - loss: 0.2006 - ssim_metric: 0.3912 - dice_coefficient: 0.6901 - val_loss: 0.2522 - val_ssim_metric: 0.7091 - val_dice_coefficient: 0.6984
Epoch 2/100
111/111 [==============================] - 30s 273ms/step - loss: 0.1159 - ssim_metric: 0.6989 - dice_coefficient: 0.7038 - val_loss: 0.2384 - val_ssim_metric: 0.7276 - val_dice_coefficient: 0.7154
Epoch 3/100
111/111 [==============================] - 31s 284ms/step - loss: 0.1008 - ssim_metric: 0.7591 - dice_coefficient: 0.7164 - val_loss: 0.1986 - val_ssim_metric: 0.7676 - val_dice_coefficient: 0.7290
Epoch 4/100
111/111 [==============================] - 32s 288ms/step - loss: 0.0920 - ssim_metric: 0.7982 - dice_coefficient: 0.7285 - val_loss: 0.1389 - val_ssim_metric: 0.8306 - val_dice_coefficient: 0.7410
Epoch 5/100
111/111 [==============================] - 32s 291ms/step - loss: 0.0856 - ssim_metric: 0.8266 - dice_coefficient: 0.7404 - val_loss: 0.0963 - val_ssim_metric: 

In [19]:

# model2.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss=custom_loss,
#     metrics=[ssim_metric, dice_coefficient]
# )

# # Train the model
# history = model2.fit(
#     train_noisy2, train_clean2,
#     validation_data=(val_noisy2, val_clean2),
#     batch_size=8,
#     epochs=100,
#     verbose=1
# )



In [20]:

# model3.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss=custom_loss,
#     metrics=[ssim_metric, dice_coefficient]
# )


# # Train the model
# history = model3.fit(
#     train_noisy3, train_clean3,
#     validation_data=(val_noisy3, val_clean3),
#     batch_size=8,
#     epochs=100,
#     verbose=1
# )



In [21]:

# model4.compile(
#     optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
#     loss=custom_loss,
#     metrics=[ssim_metric, dice_coefficient]
# )

# # Train the model
# history = model4.fit(
#     train_noisy4, train_clean4,
#     validation_data=(val_noisy4, val_clean4),
#     batch_size=8,
#     epochs=100,
#     verbose=1
# )



In [23]:
#model1.save("model1.h5")
# model2.save("model2.h5")
# model3.save("model3.h5")
# model4.save("model4.h5")


/home/user/anaconda3/envs/nn_env/lib/python3.8/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [24]:
custom_objects = {
    # STN module
    "SpatialTransformer": SpatialTransformer,

    # U-Net or your complete model builder
    "unet_with_stn": unet_with_stn,

    # Losses
    "dice_loss": dice_loss,
    "focal_loss": focal_loss,
    "ms_ssim_loss": ms_ssim_loss,
    "custom_loss": custom_loss,

    # Metrics
    "ssim_metric": ssim_metric,
    "dice_coefficient": dice_coefficient,
}


In [27]:
# from tensorflow.keras.models import load_model
# opt1 = tf.keras.optimizers.Adam()
# opt2 = tf.keras.optimizers.Adam()
# opt3 = tf.keras.optimizers.Adam()
# opt4 = tf.keras.optimizers.Adam()

# model1 = load_model("model1.h5", custom_objects=custom_objects, compile=False)
# model2 = load_model("model2.h5", custom_objects=custom_objects, compile=False)
# model3 = load_model("model3.h5", custom_objects=custom_objects, compile=False)
# model4 = load_model("model4.h5", custom_objects=custom_objects, compile=False)


In [26]:
#Freeze the experts

for m in [model1, model2, model3, model4]:
    m.trainable = False


In [28]:
import tensorflow as tf
from tensorflow.keras import layers, Model

# ----------------------------
# Gating Network (latent features)
# ----------------------------
def build_gating_network(input_shape, num_experts=4):
    inp = layers.Input(shape=input_shape)

    # Extract latent features
    x = layers.Conv2D(16, 3, activation='relu', padding='same')(inp)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(x)
    x = layers.GlobalAveragePooling2D()(x)

    # Softmax weights for each expert
    gate = layers.Dense(num_experts, activation='softmax', name="router")(x)

    return Model(inp, gate, name="gating_network")


# ----------------------------
# Mixture-of-Experts with soft training + hard inference
# ----------------------------
def build_moe_model(input_shape, experts, hard_inference=False):
    """
    Build a Mixture-of-Experts (MoE) model with soft training and optional hard inference.

    Args:
        input_shape: tuple, shape of input image (H, W, C)
        experts: list of pretrained, frozen models [model1, model2, ...]
        hard_inference: bool, if True, model outputs hard-selected expert; else weighted sum

    Returns:
        Keras Model
    """
    num_experts = len(experts)
    inp = layers.Input(shape=input_shape)

    # Compute gate weights from latent features
    gate = build_gating_network(input_shape, num_experts=num_experts)(inp)

    # Expert outputs
    expert_outputs = [expert(inp) for expert in experts]  # list of tensors

    # Stack outputs along new axis: [batch, num_experts, H, W, C]
    expert_stack = tf.stack(expert_outputs, axis=1)

    # ----- Soft output (differentiable) -----
    gate_exp = tf.reshape(gate, (-1, num_experts, 1, 1, 1))
    soft_output = tf.reduce_sum(gate_exp * expert_stack, axis=1)

    # ----- Hard output (winner-takes-all) -----
    best_idx = tf.argmax(gate, axis=1)  # [batch]
    hard_output = tf.gather(expert_stack, best_idx, axis=1, batch_dims=1)

    # Select output based on mode
    final_output = hard_output if hard_inference else soft_output

    return Model(inputs=inp, outputs=final_output, name="moe_model")


In [ ]:
input_shape = (425, 550, 1)   
#Create the MoE model
experts = [model1, model2, model3, model4]

moe_model = build_moe_model(input_shape, experts, hard_inference=False)


In [ ]:
from tensorflow.keras.optimizers import Adam

# Define optimizer with gradient clipping
optimizer = Adam(learning_rate=0.0003)


moe_model.compile(
    optimizer=optimizer,
    loss=custom_loss,
    metrics=[ssim_metric, dice_coefficient]
)


In [ ]:
moe_model.summary()

In [ ]:
history = moe_model.fit(
    train_noisy1, train_clean1,
    validation_data=(val_noisy1, val_clean1),
    batch_size=32,
    epochs=5,
    verbose=1
)
moe_model.save("moe_final_model.h5")


In [32]:
# from tensorflow.keras.models import load_model

# model = load_model("moe_final_model.h5")

In [ ]:
from tensorflow.keras.models import Model
import tensorflow as tf

# Hard inference
inp = moe_model.input
gate = moe_model.get_layer("gating_network")(inp)  # get gate output
expert_outputs = [expert(inp) for expert in [model1, model2, model3, model4]]
expert_stack = tf.stack(expert_outputs, axis=1)
best_idx = tf.argmax(gate, axis=1)
hard_output = tf.gather(expert_stack, best_idx, axis=1, batch_dims=1)

inference_model = Model(inputs=inp, outputs=hard_output)
hard_validation_images = inference_model.predict(val_noisy1)


In [ ]:
plt.imshow(hard_validation_images[5], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
plt.imshow(val_noisy1[5], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
plt.imshow(val_clean1[5], cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
import numpy as np
from skimage.metrics import structural_similarity as ssim

ssim_scores = []
dice_scores = []

for img, ref in zip(hard_validation_images, val_clean1):
    # --- Normalize or ensure binary if segmentation masks ---
    img = np.clip(img, 0, 1)
    ref = np.clip(ref, 0, 1)

    # --- SSIM Calculation ---
    h, w = img.shape[:2]
    win_size = 7
    if min(h, w) < win_size:
        win_size = min(h, w)
        if win_size % 2 == 0:
            win_size -= 1
    score_ssim = ssim(img, ref, channel_axis=-1, win_size=win_size, data_range=1.0)
    ssim_scores.append(score_ssim)

    # --- Dice Coefficient Calculation ---
    # Threshold if not binary (for grayscale or normalized float images)
    img_bin = (img > 0.5).astype(np.float32)
    ref_bin = (ref > 0.5).astype(np.float32)

    intersection = np.sum(img_bin * ref_bin)
    dice = (2. * intersection) / (np.sum(img_bin) + np.sum(ref_bin) + 1e-8)
    dice_scores.append(dice)

# --- Results ---
average_ssim = np.mean(ssim_scores)
average_dice = np.mean(dice_scores)

print(f"Average SSIM: {average_ssim:.4f}")
print(f"Average Dice Score: {average_dice:.4f}")
